# TADC-SBM: a Time-varying, Attributed, Degree-Corrected Stochastic Block Model

[![License](https://img.shields.io/github/license/nelsonaloysio/tadcsbm)](https://github.com/nelsonaloysio/tadcsbm/blob/main/LICENSE)
[![PDF](https://img.shields.io/badge/pdf-Paper-red)](https://nelsonaloysio.github.io/files/tadcsbm2025.pdf)
[![DOI](https://img.shields.io/badge/doi-10.1109/ISCC65549.2025.11326334-blue)](https://doi.org/10.1109/ISCC65549.2025.11326334)
[![GitHub](https://img.shields.io/badge/git-Code-white)](https://github.com/nelsonaloysio/tadcsbm)

In [ ]:
# for Google Colab
!pip install tadc-sbm --index-url https://test.pypi.org/simple
!pip install -q condacolab
import condacolab
condacolab.install()
!conda install -y -c conda-forge graph-tool

In [ ]:
from collections import Counter

import networkx as nx
import networkx_temporal as tx
import numpy as np

from tadcsbm import (
    tadcsbm_simulator,
    generate_block_matrix,
    generate_transition_matrix,
    # generate_degree_vector,
    generate_community_vector,
    gt_to_nx,
)

## Overview

This example generates the TADC-SBM graph sequence used in the paper, where nodes are assigned to communities and edges are generated according to the degree-corrected stochastic block model (DC-SBM) with time-varying community structure and fixed attributes. The generated graph sequence is available as a [graph-tool](https://graph-tool.skewed.de/) graph object, and node features are stored as NumPy arrays. The network is also saved to disk in a compressed format with the [NetworkX-Temporal](https://networkx-temporal.readthedocs.io) library. Available formats depend on the current NetworkX version, with added support for the legacy [`gdf`](https://networkx-gdf.readthedocs.io) format.

In [ ]:
output_dir = "output"   # Directory to save output files (graphs and features).
output_ext = "graphml"  # Format to save graph files (e.g., "graphml", "gexf", "gdf", etc.).

Main parameters for the TADC-SBM simulation are defined below. Modify these to generate different types of graphs; default parameters are set to match the graph sequence used in the paper.

In [ ]:
k = 8        # Number of communities.
t = 8        # Number of snapshots.
n = 1024     # Number of vertices.
e = 10240    # Number of edges.
eta = 1.0    # Community stability rate, where 1 means nodes never change communities.
gamma = 0    # If 0, node transition probabilities depend on current community affiliation.
beta = 1.0   # Edge sampling rate in [0, 1], where 1 means all edges are included in the graph.

The generated network may be used for benchmarking dynamic graph learning algorithms, such as node classification, link prediction, and community detection, under controlled experimentation settings. Nodes and edges may both be (optionally) assigned attributes with different signal-to-noise ratios and hierarchical (nested, overlapping) community structures.

In [ ]:
feature_dim = 32                 # Dimension of node features.
feature_center_distance = 6.0    # Distance between centroids.
feature_cluster_variance = 1.0   # Variance of feature clusters.
edge_feature_dim = 32            # Dimension of edge features.
edge_center_distance = 6.0       # Distance between edge feature centroids.
edge_cluster_variance = 1.0      # Variance of edge feature clusters.
reverse_snapshot_order = True    # If True, snapshot indices are inverted.
uniform_all = False              # If False, discard self-transition probabilities.

Display block matrix $\mathbf{B}$, transition probability matrix $\mathbf{\tau}$, and community assignments $\mathbf{Z}$ for the first snapshot.

In [ ]:
mat = generate_block_matrix(k)
mat

In [ ]:
tau = generate_transition_matrix(k, eta, uniform_all=False)
tau

In [ ]:
z = generate_community_vector(n, k, shuffle=False)
z

Generate TADC-SBM graph sequence with the specified parameters.

In [ ]:
sbm = tadcsbm_simulator(
    snapshots=t,
    num_vertices=n,
    num_edges=e,
    pi=[v/len(z) for k, v in Counter(z).items()],
    prop_mat=mat,
    tau_mat=tau,
    num_feature_groups=k,
    feature_dim=feature_dim
    feature_center_distance=feature_center_distance
    feature_cluster_variance=feature_cluster_variance
    edge_feature_dim=edge_feature_dim
    edge_center_distance=edge_center_distance
    edge_cluster_variance=edge_cluster_variance
    fixed_probabilities=gamma
    reverse_snapshot_order=reverse_snapshot_order
    edge_sampling_rate=beta
)

Compose graph-tool graphs as a single NetworkX-Temporal multigraph.

In [ ]:
TG = tx.from_snapshots([gt_to_nx(graph, time=t) for t, graph in enumerate(sbm.graph)])
print(TG)

Set node attributes for community memberships and save to disk.

In [ ]:
list(nx.set_node_attributes(G, {v: y for v, y in zip(G.nodes(), sbm.graph_memberships)}, "y") for G in TG)

Save node and edge features as NumPy arrays.

In [ ]:
if feature_dim > 0:
    print("Saving node features...", end="\r")
    np.save(f"{output_dir}/features_node.npy", sbm.node_features1)
if edge_feature_dim > 0:
    print("Saving edge features...", end="\r")
    np.save(f"{output_dir}/features_edge.npy", sbm.edge_features)

Save the composed graph to disk. NOTE: node and edge attributes are not saved with graph file, but timestamps and community assignments are preserved.

In [ ]:
tx.write_graph(TG, f"{output_dir}/graphs.{output_ext}.zip")
tx.write_graph(TG.to_static(), f"{output_dir}/graph.{output_ext}.zip")

___

Inspect the transition probability function $P(A \vert \mathbf{B}, \mathbf{T})$ for a single snapshot ($t>0$) to verify that the generated graph is consistent with the specified block matrix $\mathbf{B}$ and transition probabilities $\mathbf{\tau}$, with node assignments $\mathbf{Z}$ as a hidden parameter not used by the transition function, which only depends on the current block and transition matrices.

In [ ]:
from tadcsbm.simulations.sbm_simulator import _TransitionNodeMemberships
_TransitionNodeMemberships(sbm.graph_memberships, tau)